In [1]:
#cell1
# Setup imports

import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from google.colab import drive
from IPython.display import display

In [2]:
#cell2
# Mount Google Drive silently

import io
import contextlib

with contextlib.redirect_stdout(io.StringIO()):
    drive.mount("/content/drive", force_remount=False)

In [3]:
# Cell 3
# Define BM25 file paths, dataset names, model names, and output directory

BASE_DIR = Path("/content/drive/MyDrive/final_project/BM25/answer")
OUTPUT_DIR = Path("/content/drive/MyDrive/final_project/BM25/f1_results")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANSWER_FILES = [
    {
        "dataset": "hotpotqa",
        "model": "gemma4",
        "file_name": "hotpotqa_bm25_gemma4_answers.json",
        "file_path": BASE_DIR / "hotpotqa_bm25_gemma4_answers.json",
    },
    {
        "dataset": "2wikimultihopqa",
        "model": "gemma4",
        "file_name": "2wikimultihopqa_bm25_gemma4_answers.json",
        "file_path": BASE_DIR / "2wikimultihopqa_bm25_gemma4_answers.json",
    },
    {
        "dataset": "hotpotqa",
        "model": "gpt-oss-120b",
        "file_name": "hotpotqa_bm25_gpt_oss_120b_answers.json",
        "file_path": BASE_DIR / "hotpotqa_bm25_gpt_oss_120b_answers.json",
    },
    {
        "dataset": "2wikimultihopqa",
        "model": "gpt-oss-120b",
        "file_name": "2wikimultihopqa_bm25_gpt_oss_120b_answers.json",
        "file_path": BASE_DIR / "2wikimultihopqa_bm25_gpt_oss_120b_answers.json",
    },
    {
        "dataset": "hotpotqa",
        "model": "qwen3.5",
        "file_name": "hotpotqa_qwen3.5_bm25_answers.json",
        "file_path": BASE_DIR / "hotpotqa_qwen3.5_bm25_answers.json",
    },
    {
        "dataset": "2wikimultihopqa",
        "model": "qwen3.5",
        "file_name": "2wikimultihopqa_qwen3.5_bm25_answers.json",
        "file_path": BASE_DIR / "2wikimultihopqa_qwen3.5_bm25_answers.json",
    },
]

# Check that all expected BM25 files exist before evaluation
missing_files = [str(item["file_path"]) for item in ANSWER_FILES if not item["file_path"].exists()]

if missing_files:
    print("The following BM25 files were not found:")
    for path in missing_files:
        print(path)
    raise FileNotFoundError("Some BM25 answer files are missing. Please check BASE_DIR and file names.")
else:
    print("All BM25 answer files were found successfully.")
    print(f"Input directory: {BASE_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")

All BM25 answer files were found successfully.
Input directory: /content/drive/MyDrive/final_project/BM25/answer
Output directory: /content/drive/MyDrive/final_project/BM25/f1_results


In [4]:
#cell4
# Normalize text and tokenize answers

def normalize_text(text):
    """
    Basic normalization for token-level comparison.
    Lowercase, remove punctuation, and normalize spaces.
    """
    if text is None:
        text = ""

    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.casefold()

    chars = []
    for ch in text:
        if unicodedata.category(ch).startswith("P"):
            chars.append(" ")
        else:
            chars.append(ch)

    text = "".join(chars)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    """
    Convert answer text to a set of tokens.
    This keeps the same set-overlap logic as the previous notebook.
    """
    normalized = normalize_text(text)
    if not normalized:
        return set()
    return set(normalized.split())

In [5]:
#cell5
# Compute Precision, Recall, and F1 for one example

def compute_token_f1(predicted_answer, ground_truth_answer):
    """
    Compute token-level Precision, Recall, and F1.
    """
    pred_tokens = tokenize(predicted_answer)
    gt_tokens = tokenize(ground_truth_answer)

    if len(pred_tokens) == 0 and len(gt_tokens) == 0:
        return 1.0, 1.0, 1.0

    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return 0.0, 0.0, 0.0

    overlap = pred_tokens.intersection(gt_tokens)

    precision = len(overlap) / len(pred_tokens)
    recall = len(overlap) / len(gt_tokens)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = (2 * precision * recall) / (precision + recall)

    return precision, recall, f1

In [6]:
#cell6
# Load one JSON file and compute row-level scores

def load_json_file(file_path):
    """
    Load a JSON answer file.
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"Expected a list of records in: {file_path}")

    return data


def evaluate_file(dataset_name, model_name, file_name, file_path):
    """
    Evaluate all questions for one dataset-model file.
    """
    data = load_json_file(file_path)
    rows = []

    for idx, item in enumerate(data):
        gt = item.get("gt", "")
        response = item.get("response", "")
        question_type = item.get("type", "unknown")

        precision, recall, f1 = compute_token_f1(
            predicted_answer=response,
            ground_truth_answer=gt
        )

        rows.append({
            "dataset": dataset_name,
            "model": model_name,
            "file_name": file_name,
            "row_index": idx,
            "source_index": item.get("source_index", idx),
            "type": question_type if question_type is not None else "unknown",
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "question": item.get("question", ""),
            "gt": gt,
            "response": response,
        })

    return pd.DataFrame(rows)

In [7]:
#cell7
# Evaluate all files while keeping dataset, model, and file identity separated

def evaluate_all_files(answer_files):
    """
    Evaluate every JSON file and combine row-level results.
    Each row keeps dataset, model, and file_name to prevent mixing results.
    """
    all_dfs = []

    for file_info in answer_files:
        file_df = evaluate_file(
            dataset_name=file_info["dataset"],
            model_name=file_info["model"],
            file_name=file_info["file_name"],
            file_path=file_info["file_path"],
        )
        all_dfs.append(file_df)

    return pd.concat(all_dfs, ignore_index=True)

In [8]:
#cell8
# Build overall and per-type summaries for each separate file

def build_summary(scores_df):
    """
    Create overall and per-type macro summaries.
    Results are grouped by dataset, model, and file_name so files do not get mixed.
    """
    group_cols = ["dataset", "model", "file_name"]

    overall_df = (
        scores_df
        .groupby(group_cols, as_index=False)
        .agg(
            n_questions=("f1", "size"),
            precision_macro=("precision", "mean"),
            recall_macro=("recall", "mean"),
            f1_macro=("f1", "mean"),
        )
    )
    overall_df["type"] = "overall"

    type_df = (
        scores_df
        .groupby(group_cols + ["type"], as_index=False)
        .agg(
            n_questions=("f1", "size"),
            precision_macro=("precision", "mean"),
            recall_macro=("recall", "mean"),
            f1_macro=("f1", "mean"),
        )
    )

    summary_df = pd.concat([overall_df, type_df], ignore_index=True)

    summary_df["f1_percent"] = summary_df["f1_macro"] * 100

    dataset_order = ["hotpotqa", "2wikimultihopqa"]
    model_order = ["qwen3.5", "gpt-oss-120b", "gemma4"]

    summary_df["dataset"] = pd.Categorical(
        summary_df["dataset"],
        categories=dataset_order,
        ordered=True
    )

    summary_df["model"] = pd.Categorical(
        summary_df["model"],
        categories=model_order,
        ordered=True
    )

    summary_df["type_sort"] = summary_df["type"].apply(
        lambda x: "000_overall" if x == "overall" else str(x)
    )

    summary_df = (
        summary_df
        .sort_values(["dataset", "model", "file_name", "type_sort"])
        .drop(columns=["type_sort"])
        .reset_index(drop=True)
    )

    column_order = [
        "dataset",
        "model",
        "file_name",
        "type",
        "n_questions",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "f1_percent",
    ]
    summary_df = summary_df[column_order]

    numeric_cols = ["precision_macro", "recall_macro", "f1_macro", "f1_percent"]
    summary_df[numeric_cols] = summary_df[numeric_cols].round(6)

    return summary_df

In [9]:
# Cell 9
# Run BM25 F1 evaluation for HotpotQA only and save results separately

hotpotqa_answer_files = [
    item for item in ANSWER_FILES
    if item["dataset"] == "hotpotqa"
]

hotpotqa_scores_df = evaluate_all_files(hotpotqa_answer_files)
hotpotqa_summary_df = build_summary(hotpotqa_scores_df)

hotpotqa_scores_path = OUTPUT_DIR / "hotpotqa_bm25_f1_row_scores.csv"
hotpotqa_summary_csv_path = OUTPUT_DIR / "hotpotqa_bm25_f1_summary.csv"
hotpotqa_summary_json_path = OUTPUT_DIR / "hotpotqa_bm25_f1_summary.json"

hotpotqa_scores_df.to_csv(hotpotqa_scores_path, index=False, encoding="utf-8-sig")
hotpotqa_summary_df.to_csv(hotpotqa_summary_csv_path, index=False, encoding="utf-8-sig")
hotpotqa_summary_df.to_json(
    hotpotqa_summary_json_path,
    orient="records",
    force_ascii=False,
    indent=2
)

print("=" * 100)
print("HotpotQA BM25 F1 Summary")
print("=" * 100)

display(hotpotqa_summary_df)

print("\nSaved files:")
print(f"Row-level scores: {hotpotqa_scores_path}")
print(f"Summary CSV:      {hotpotqa_summary_csv_path}")
print(f"Summary JSON:     {hotpotqa_summary_json_path}")

HotpotQA BM25 F1 Summary


,dataset,model,file_name,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,hotpotqa,qwen3.5,hotpotqa_qwen3.5_bm25_answers.json,overall,1000,0.465648,0.578631,0.477081,47.708100
1,hotpotqa,qwen3.5,hotpotqa_qwen3.5_bm25_answers.json,bridge,700,0.509085,0.528757,0.499969,49.996920
2,hotpotqa,qwen3.5,hotpotqa_qwen3.5_bm25_answers.json,comparison,300,0.364294,0.695005,0.423675,42.367519
3,hotpotqa,gpt-oss-120b,hotpotqa_bm25_gpt_oss_120b_answers.json,overall,1000,0.461543,0.508704,0.457959,45.795914
4,hotpotqa,gpt-oss-120b,hotpotqa_bm25_gpt_oss_120b_answers.json,bridge,700,0.518990,0.526544,0.505430,50.542992
5,hotpotqa,gpt-oss-120b,hotpotqa_bm25_gpt_oss_120b_answers.json,comparison,300,0.327500,0.467078,0.347194,34.719400
6,hotpotqa,gemma4,hotpotqa_bm25_gemma4_answers.json,overall,1000,0.416471,0.460559,0.414961,41.496080
7,hotpotqa,gemma4,hotpotqa_bm25_gemma4_answers.json,bridge,700,0.423207,0.451755,0.421836,42.183629
8,hotpotqa,gemma4,hotpotqa_bm25_gemma4_answers.json,comparison,300,0.400754,0.481103,0.398918,39.891799



Saved files:
Row-level scores: /content/drive/MyDrive/final_project/BM25/f1_results/hotpotqa_bm25_f1_row_scores.csv
Summary CSV:      /content/drive/MyDrive/final_project/BM25/f1_results/hotpotqa_bm25_f1_summary.csv
Summary JSON:     /content/drive/MyDrive/final_project/BM25/f1_results/hotpotqa_bm25_f1_summary.json


In [10]:
# Cell 10
# Run BM25 F1 evaluation for 2WikiMultiHopQA only and save results separately

twowiki_answer_files = [
    item for item in ANSWER_FILES
    if item["dataset"] == "2wikimultihopqa"
]

twowiki_scores_df = evaluate_all_files(twowiki_answer_files)
twowiki_summary_df = build_summary(twowiki_scores_df)

twowiki_scores_path = OUTPUT_DIR / "2wikimultihopqa_bm25_f1_row_scores.csv"
twowiki_summary_csv_path = OUTPUT_DIR / "2wikimultihopqa_bm25_f1_summary.csv"
twowiki_summary_json_path = OUTPUT_DIR / "2wikimultihopqa_bm25_f1_summary.json"

twowiki_scores_df.to_csv(twowiki_scores_path, index=False, encoding="utf-8-sig")
twowiki_summary_df.to_csv(twowiki_summary_csv_path, index=False, encoding="utf-8-sig")
twowiki_summary_df.to_json(
    twowiki_summary_json_path,
    orient="records",
    force_ascii=False,
    indent=2
)

print("=" * 100)
print("2WikiMultiHopQA BM25 F1 Summary")
print("=" * 100)

display(twowiki_summary_df)

print("\nSaved files:")
print(f"Row-level scores: {twowiki_scores_path}")
print(f"Summary CSV:      {twowiki_summary_csv_path}")
print(f"Summary JSON:     {twowiki_summary_json_path}")

2WikiMultiHopQA BM25 F1 Summary


,dataset,model,file_name,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_bm25_answers.json,overall,1000,0.349574,0.470254,0.373409,37.340905
1,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_bm25_answers.json,bridge_comparison,250,0.320800,0.366067,0.327146,32.714603
2,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_bm25_answers.json,comparison,250,0.431438,0.835333,0.528448,52.844791
3,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_bm25_answers.json,compositional,250,0.179181,0.202333,0.179638,17.963810
4,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_bm25_answers.json,inference,250,0.466876,0.477281,0.458404,45.840417
5,2wikimultihopqa,gpt-oss-120b,2wikimultihopqa_bm25_gpt_oss_120b_answers.json,overall,1000,0.274500,0.356570,0.290856,29.085627
6,2wikimultihopqa,gpt-oss-120b,2wikimultihopqa_bm25_gpt_oss_120b_answers.json,bridge_comparison,250,0.069533,0.100000,0.075333,7.533333
7,2wikimultihopqa,gpt-oss-120b,2wikimultihopqa_bm25_gpt_oss_120b_answers.json,comparison,250,0.479905,0.762857,0.550089,55.008947
8,2wikimultihopqa,gpt-oss-120b,2wikimultihopqa_bm25_gpt_oss_120b_answers.json,compositional,250,0.149000,0.162867,0.147404,14.740433
9,2wikimultihopqa,gpt-oss-120b,2wikimultihopqa_bm25_gpt_oss_120b_answers.json,inference,250,0.399562,0.400557,0.390598,39.059794



Saved files:
Row-level scores: /content/drive/MyDrive/final_project/BM25/f1_results/2wikimultihopqa_bm25_f1_row_scores.csv
Summary CSV:      /content/drive/MyDrive/final_project/BM25/f1_results/2wikimultihopqa_bm25_f1_summary.csv
Summary JSON:     /content/drive/MyDrive/final_project/BM25/f1_results/2wikimultihopqa_bm25_f1_summary.json
